# Assignment 2: Milestone I Natural Language Processing
## Task 1. Basic Text Pre-processing
#### Student Name - Student ID: 
- Nguyen Trong Tien - s3978616
- Nguyen Hai Long - s4002308
- Vo Nguyen Bao Ngoc - s3975091
- Truong Gia Hy - s4053650





## Contents

1. [Introduction](#introduction)
2. [Environment & Imports](#environment-import)
3. [Data Loading](#data-loading)
- 3.1. [Read and print data](#read-and-print-data)
- 3.2. [Pre-processing data](#pre-processing-data)
4. [Export Vocabulary & Processed Dataset](#export-vocabulary-processed-dataset)


## 1. Introduction <a id="introduction"></a>

In this milestone, the goal is to apply Natural Language Processing (NLP) to a collection of clothing reviews in order to predict product recommendations. The tasks include pre-processing the review text (tokenization, stopword removal, and frequency-based filtering), generating feature representations (Bag-of-Words and word embeddings), and building machine learning models to classify whether a review  indicates a recommendation. Finally, the models will be evaluated and analyzed to determine their effectiveness.

## 2. Environment & Import <a id="environment-import"></a>
### Environment: : Python 3 and Jupyter notebook

Libraries used: 
* pandas
* re
* numpy
* matplotlib
* nltk
* spellchecker
* TextBlob
* Counter
* wordnet
* RegexpTokenizer
* WordNetLemmatizer

### Importing libraries 

In [1]:
# Import neccessary libraries for this assessment:
import numpy as np
import matplotlib.pyplot as plt
import os
import re
import zipfile
from textblob import TextBlob
from collections import Counter
from pathlib import Path
from nltk.tokenize import RegexpTokenizer
from nltk.stem import WordNetLemmatizer
from spellchecker import SpellChecker
from nltk.corpus import wordnet
import nltk
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')
nltk.download('averaged_perceptron_tagger')
nltk.download('averaged_perceptron_tagger_eng')
from collections import Counter
import pandas as pd


[nltk_data] Downloading package punkt to /Users/vongoc/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/vongoc/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /Users/vongoc/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /Users/vongoc/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /Users/vongoc/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /Users/vongoc/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!


## 3. Loading data <a id="data-loading"></a>

### 3.1. Read and print data <a id="read-and-print-data"></a>

In [2]:
# Check data

df = pd.read_csv('assignment3.csv')
df.info()
df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 19662 entries, 0 to 19661
Data columns (total 10 columns):
 #   Column                   Non-Null Count  Dtype 
---  ------                   --------------  ----- 
 0   Clothing ID              19662 non-null  int64 
 1   Age                      19662 non-null  int64 
 2   Title                    19662 non-null  object
 3   Review Text              19662 non-null  object
 4   Rating                   19662 non-null  int64 
 5   Recommended IND          19662 non-null  int64 
 6   Positive Feedback Count  19662 non-null  int64 
 7   Division Name            19662 non-null  object
 8   Department Name          19662 non-null  object
 9   Class Name               19662 non-null  object
dtypes: int64(5), object(5)
memory usage: 1.5+ MB


,Clothing ID,Age,Title,Review Text,Rating,Recommended IND,Positive Feedback Count,Division Name,Department Name,Class Name
0,1077,60,Some major design flaws,I had such high hopes for this dress and reall...,3,0,0,General,Dresses,Dresses
1,1049,50,My favorite buy!,"I love, love, love this jumpsuit. it's fun, fl...",5,1,0,General Petite,Bottoms,Pants
2,847,47,Flattering shirt,This shirt is very flattering to all due to th...,5,1,6,General,Tops,Blouses
3,1080,49,Not for the very petite,"I love tracy reese dresses, but this one is no...",2,0,4,General,Dresses,Dresses
4,858,39,Cagrcoal shimmer fun,I aded this in my basket at hte last mintue to...,5,1,1,General Petite,Tops,Knits


### Insight — Missing values / schema
- **What we checked:** Null counts by column and overall completeness.
- **What to look for:** Features with heavy missingness (>20%) merit imputation strategy notes or removal; MCAR vs MAR hints from patterns across columns.

### **3.2 Pre-processing data** <a id="pre-processing-data"></a>
##### Tokenize & Normalize Review Text  <a id='sec-clean'></a>
Prepare raw review text for modeling by turning it into tokens while avoiding information loss that could hurt downstream classification.

First, we tokenize Review Text into tokens

In [3]:
# Tokenizing clothing review

# define tokenizer and keeps meaningful forms
tokenizer = RegexpTokenizer(r"[a-zA-Z]+(?:[-'][a-zA-Z]+)?")

# Handle missing text and tokenize the review text + lowercasing and removing words with length < 2
df["tokens"] = (
    df["Review Text"]
      .fillna("") # replace NaN with empty string
      .astype(str)
      .apply(lambda text: [token.lower() for token in tokenizer.tokenize(text) if len(token) > 1])
)

# quick spot-checks (delete when done)
print(df["tokens"].head(3).tolist())
assert isinstance(df["tokens"].iloc[0], list)

[['had', 'such', 'high', 'hopes', 'for', 'this', 'dress', 'and', 'really', 'wanted', 'it', 'to', 'work', 'for', 'me', 'initially', 'ordered', 'the', 'petite', 'small', 'my', 'usual', 'size', 'but', 'found', 'this', 'to', 'be', 'outrageously', 'small', 'so', 'small', 'in', 'fact', 'that', 'could', 'not', 'zip', 'it', 'up', 'reordered', 'it', 'in', 'petite', 'medium', 'which', 'was', 'just', 'ok', 'overall', 'the', 'top', 'half', 'was', 'comfortable', 'and', 'fit', 'nicely', 'but', 'the', 'bottom', 'half', 'had', 'very', 'tight', 'under', 'layer', 'and', 'several', 'somewhat', 'cheap', 'net', 'over', 'layers', 'imo', 'major', 'design', 'flaw', 'was', 'the', 'net', 'over', 'layer', 'sewn', 'directly', 'into', 'the', 'zipper', 'it'], ['love', 'love', 'love', 'this', 'jumpsuit', "it's", 'fun', 'flirty', 'and', 'fabulous', 'every', 'time', 'wear', 'it', 'get', 'nothing', 'but', 'great', 'compliments'], ['this', 'shirt', 'is', 'very', 'flattering', 'to', 'all', 'due', 'to', 'the', 'adjustable

##### **Text Spelling Correction for Tokenized Data with TextBlob**
Here we try to reduce words having typos and incorrect spellings in reviews by correcting each token to its most probable approriate word

Define functions to initialize TextBlob and apply it to the tokens (this would take time to run)

In [4]:
def correct_text(text):
    return str(TextBlob(text).correct())

def correct_tokens(token_list):
    corrected_tokens = []
    for token in token_list:
        corrected_token = correct_text(token)
        corrected_tokens.append(corrected_token)
    return corrected_tokens

df["tokens"] = df["tokens"].apply(correct_tokens)

##### Spell Checking & Token Cleaning for Tokenized Reviews
Reduce vocabulary noise by lowercasing and removing nulls, then correct misspellings again token-by-token using SpellChecker

Define SpellChecker and apply it to tokens, we use both TextBlob and SpellChecker to reduce as much typo error as possible (this would take time to run)

In [ ]:
spell = SpellChecker()


# Clean tokens
def clean_tokens(tokens):
    if not isinstance(tokens, list):
        return []
    return [str(t).lower() for t in tokens if t is not None]

# Spell checking
def correct_tokens(tokens):
    corrected = []
    for word in tokens:
        if word in spell.unknown([word]):
            suggestion = spell.correction(word)
            corrected.append(suggestion if suggestion is not None else word)
        else:
            corrected.append(word)
    return corrected


# Apply pipeline
df["tokens"] = df["tokens"].apply(clean_tokens)
df["tokens"] = df["tokens"].apply(correct_tokens)

##### Stopword Removal
Remove words provided in stopwords_en.txt

Here we read the file and remove words from that file

In [ ]:
# Remove stopwords using the provided stop words list (i.e., stopwords_en.txt). It is located inside the same downloaded folder. 
# Find file stopwords in data
stop_path = Path("stopwords_en.txt")

# Load stopwords
with open(stop_path, "r", encoding="utf-8") as f:
    stop_words = {line.strip() for line in f if line.strip()}

print(f"[Q5] Loaded {len(stop_words)} stopwords from {stop_path}")

# Remove stopwords (Compare lowercase)
def remove_stopwords(tokens, stopset):
    return [t for t in tokens if t.lower() not in stopset]

df["tokens"] = df["tokens"].apply(remove_stopwords, stopset=stop_words)

# Print to check
print(df[["tokens"]].head(2))

[Q5] Loaded 570 stopwords from stopwords_en.txt
                                              tokens
0  [high, hopes, dress, wanted, work, initially, ...
1  [love, love, love, jumpsuit, fun, flirty, fabu...


##### Token Pruning  
Remove the word that appears only once in the document collection, based on term frequency.  
Remove the top 20 most frequent words based on document frequency.

We read the tokens' term frequencies and keep only those higher than 1

In [ ]:
# Remove the word that appears only once in the document collection, based on term frequency.
term_freq = pd.Series(np.concatenate(df["tokens"].values)).value_counts()
 
df["tokens"] = df["tokens"].apply(lambda tokens: [token for token in tokens if term_freq[token] > 1])

# Double check
term_freq = pd.Series(np.concatenate(df["tokens"].values)).value_counts()
term_freq

dress          9334
size           7860
love           7722
fit            6582
top            6542
               ... 
upsize            2
relaxed-fit       2
undressed         2
connected         2
crosswrap         2
Name: count, Length: 7549, dtype: int64

Similarly, we count the document frequencies and remove top 20 highest

In [ ]:
# Remove the top 20 most frequent words based on document frequency. 
# Count document frequency (each word counts only once per doc)
doc_freq = Counter()
for tokens in df["tokens"]:
    unique_tokens = set(tokens)
    doc_freq.update(unique_tokens)

# Get the top 20 most frequent words
top_20_words = {word for word, _ in doc_freq.most_common(20)}
print("Top 20 frequent words:", top_20_words)

# Remove them from each token list
df["tokens"] = df["tokens"].apply(
    lambda tokens: [w for w in tokens if w not in top_20_words]
)


Top 20 frequent words: {'fabric', 'small', 'ordered', 'color', 'top', 'fits', 'fit', 'flattering', 'back', 'nice', 'dress', 'cute', 'soft', 'comfortable', 'perfect', 'bought', 'wear', 'love', 'great', 'size'}


#### Lematization  
Lemmatize tokens to combine relevant words with different suffixes

In [ ]:
# Creates an NLTK WordNet lemmatizer and replaces each token in every review with its lemma
lemmatizer = WordNetLemmatizer()
df["tokens"] = [[lemmatizer.lemmatize(w) for w in review] for review in df["tokens"].tolist()]

We clean the vocabulary again

In [ ]:
# Vocabulary Sanitization
def clean_vocab_token(token):
    token = token.lower().strip()
    if re.fullmatch(r"[a-z0-9]+", token) and len(token) > 2:
        return token
    return None

cleaned_tokens = df["tokens"].apply(lambda tokens: [clean_vocab_token(t) for t in tokens if t is not None])
df["tokens"] = cleaned_tokens.apply(lambda tokens: [t for t in tokens if t is not None])

In [ ]:
# Print to check the tokens column
df[["tokens"]].head(50)

,tokens
0,"[high, hope, wanted, work, initially, petite, ..."
1,"[jumpsuit, fun, flirty, fabulous, time, compli..."
2,"[shirt, due, adjustable, front, tie, length, l..."
3,"[tracy, reese, dress, petite, foot, tall, bran..."
4,"[basket, hte, person, store, pick, teh, pale, ..."
5,"[carbon, store, pick, ton, stuff, pair, skirt,..."
6,"[run, snug, bust, feminine, usual, retailer, f..."
7,"[petite, make, length, long, typically, regula..."
8,"[run, esp, zipper, area, run, typically, tight..."
9,"[find, review, written, savvy, shopper, past, ..."


##### Vocabulary Construction & Global Token Frequency
Build a vocabulary of the cleaned/processed reviews, and save it in a txt file

Flatten all tokens and construct a vocab dictionary

In [ ]:
# Flatten all tokens
all_tokens = [t for tokens in df["tokens"] for t in tokens]

# Sort vocab
unique_vocab = sorted(set(all_tokens))
vocab_dict = {word: idx for idx, word in enumerate(unique_vocab)}
vocab_dict

{'abbey': 0,
 'abby': 1,
 'abdomen': 2,
 'ability': 3,
 'abnormally': 4,
 'abo': 5,
 'abou': 6,
 'abroad': 7,
 'absolute': 8,
 'absolutely': 9,
 'absolutley': 10,
 'absolutly': 11,
 'abstract': 12,
 'absurd': 13,
 'abt': 14,
 'abundance': 15,
 'accent': 16,
 'accented': 17,
 'accenting': 18,
 'accentuate': 19,
 'accentuated': 20,
 'accentuates': 21,
 'accentuating': 22,
 'accept': 23,
 'acceptable': 24,
 'accepted': 25,
 'access': 26,
 'accessorize': 27,
 'accessorized': 28,
 'accessorizing': 29,
 'accessory': 30,
 'accident': 31,
 'accidental': 32,
 'accidentally': 33,
 'accommodate': 34,
 'accommodated': 35,
 'accommodates': 36,
 'accommodating': 37,
 'accomodate': 38,
 'accompanying': 39,
 'accomplish': 40,
 'accordian': 41,
 'account': 42,
 'accurate': 43,
 'accurately': 44,
 'acetate': 45,
 'achieve': 46,
 'acrylic': 47,
 'act': 48,
 'action': 49,
 'active': 50,
 'activewear': 51,
 'activity': 52,
 'actual': 53,
 'actuality': 54,
 'ada': 55,
 'add': 56,
 'added': 57,
 'addict': 58

## 4. Export Vocabulary & Processed Dataset <a id="export-vocabulary-processed-dataset"></a>
Save the requested information as per specification.
- vocab.txt

In [ ]:
# Saving vocab.txt (as word:idx) and processed.csv freezes the artifacts used for modeling
with open("vocab.txt", "w", encoding="utf-8") as f:
    for word, idx in vocab_dict.items():
        f.write(f"{word}:{idx}\n")

df.to_csv("processed.csv", index=False)